In [ ]:
import numpy as np
import pandas as pd
from scipy.special import comb as nchoosek
import os
import gzip
import tqdm.notebook as tqdm
import matplotlib.pyplot as plt

from natvar.helpers import gene_seq_to_array

In [ ]:
DATDIR = "../data/dataset_entero"
RESULTS_DIR = "../results/dataset_entero"

QUERIES_DIR = "../data/queries"

wtsequences_fpath = f"../data/wtsequences.csv"

SAVE_DFS = True
OUTDIR = "../out/nb_gather_results"
IMGDIR = f"{OUTDIR}/images"
OUT_DF_DIR = f"{OUTDIR}/dfs"
OUT_SEQ_DIR = f"{OUTDIR}/sequences"

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(IMGDIR, exist_ok=True)
os.makedirs(OUT_DF_DIR, exist_ok=True)
os.makedirs(OUT_SEQ_DIR, exist_ok=True)

LOAD_SAVED_DFS = False

In [ ]:
##############################################################################
##  Wildtype promoter sequences

df_wtsequences = pd.read_csv(wtsequences_fpath)

NGENES = len(df_wtsequences)
assert NGENES == 107, f"Expected 107 genes. Got {NGENES}"

gene_list = df_wtsequences['name'].values.tolist()
promoter_list = df_wtsequences['geneseq'].values.tolist()
print(f"Loaded {len(gene_list)} gene names.")

In [ ]:
##############################################################################
##  Query lists

query_list_fpath_0 = f"{QUERIES_DIR}/query_list_0_30.txt"
query_list_fpath_1 = f"{QUERIES_DIR}/query_list_130_160.txt"

query_list0 = np.genfromtxt(query_list_fpath_0, dtype=str)
query_list1 = np.genfromtxt(query_list_fpath_1, dtype=str)

QUERY_LENGTH = len(query_list0[0])
assert QUERY_LENGTH == 30, f"Expected query length of 30. Got {QUERY_LENGTH}"

In [ ]:
##############################################################################
##  Genome sizes

GENOME_LENGTH = 4.6 * 1E6

genome_lengths_fpath = f"{DATDIR}/genome_sizes/genome_lengths.npy"
contig_lengths_fpath = f"{DATDIR}/genome_sizes/contig_lengths.npy.gz"

genome_lengths = np.load(genome_lengths_fpath)
contig_lengths = np.load(
    gzip.GzipFile(contig_lengths_fpath, "r"), allow_pickle=True
)

NGENOMES = len(genome_lengths)

In [ ]:
##############################################################################
##  Results

mq_results_0_dir = f"{RESULTS_DIR}/multiquery_results_0_30"
mq_results_1_dir = f"{RESULTS_DIR}/multiquery_results_130_160"

NRESULT_FILES = len([d for d in os.listdir(mq_results_0_dir) if "results" in d])
assert NRESULT_FILES == 295, f"Expected 295 genomes. Got {NRESULT_FILES}"

In [ ]:
##############################################################################
##  Determine threshold value for calling two k-length sequences variants

TAU = 0.01

k = np.arange(QUERY_LENGTH + 1)
p = 3/4
num_exp_seqs_with_k_muts = GENOME_LENGTH * nchoosek(QUERY_LENGTH, k) \
    * p**k * (1 - p)**(QUERY_LENGTH - k)

nmut_thresh = k[num_exp_seqs_with_k_muts > TAU][0] - 1
print("Threshold:", nmut_thresh, "(inclusive)")
print(f"  E[{nmut_thresh-1}]={num_exp_seqs_with_k_muts[nmut_thresh-1]}")
print(f"  E[{nmut_thresh}]={num_exp_seqs_with_k_muts[nmut_thresh]}")
print(f"  E[{nmut_thresh+1}]={num_exp_seqs_with_k_muts[nmut_thresh+1]}")

In [ ]:
##############################################################################
##  Construct mappings from gene index to values of interest

gene_map = {i: g for i, g in enumerate(gene_list)}
gene_to_idx = {g: i for i, g in enumerate(gene_list)}

wtsequences = {i: s for i, s in enumerate(promoter_list)}
query_map0 = {i: seq for i, seq in enumerate(query_list0)}
query_map1 = {i: seq for i, seq in enumerate(query_list1)}

idx = 0
print(f"Gene {idx}: {gene_map[idx]}")
print(f"        Wildtype: {wtsequences[idx][0:40]}...{wtsequences[idx][-40:]}")
print("      Query 0-30:", query_map0[idx])
print("   Query 130-160:", query_map1[idx])

In [ ]:
if LOAD_SAVED_DFS:
    results0_by_gene = {i: [] for i in gene_map.keys()}
    results1_by_gene = {i: [] for i in gene_map.keys()}
    converters = {
        "nearest_idxs": lambda x: np.array(eval(x)),
        "location_on_contigs": lambda x: np.array(eval(x)),
        "contig_segments": eval,
    }
    for gidx in tqdm.trange(NGENES, disable=False):
        results0_by_gene[gidx] = pd.read_csv(
            f"{OUTDIR}/dataframes/df_results0_gene{gidx}.csv",
            converters=converters,
        )
        results1_by_gene[gidx] = pd.read_csv(
            f"{OUTDIR}/dataframes/df_results1_gene{gidx}.csv",
            converters=converters,
        )
else:
    assert len(os.listdir(mq_results_0_dir)) == NRESULT_FILES, \
        f"Wrong number of files. Got {len(os.listdir(mq_results_0_dir))}"
    assert len(os.listdir(mq_results_1_dir)) == NRESULT_FILES, \
        f"Wrong number of files. Got {len(os.listdir(mq_results_1_dir))}"

    results0_by_gene = {i: [] for i in gene_map.keys()}
    results1_by_gene = {i: [] for i in gene_map.keys()}
    for i in tqdm.trange(NRESULT_FILES, disable=False):
        fpath0 = f"{mq_results_0_dir}/results_{i}.tsv.gz"
        fpath1 = f"{mq_results_1_dir}/results_{i}.tsv.gz"
        df0 = pd.read_csv(fpath0, sep='\t', compression='gzip')
        df1 = pd.read_csv(fpath1, sep='\t', compression='gzip')

        for gene_idx, gene_name in gene_map.items():
            q0 = query_map0[gene_idx]
            q1 = query_map1[gene_idx]
            # Subset for rows corresponding to the gene of interest
            df0_subset = df0[df0['query_string'] == q0]
            df1_subset = df1[df1['query_string'] == q1]
            # Subset for rows where the min_distance is less than the threshold
            df0_subset = df0_subset[df0_subset['min_distance'] <= nmut_thresh]
            df1_subset = df1_subset[df1_subset['min_distance'] <= nmut_thresh]
            # Save desired columns
            cols_to_store = [
                'genome_fpath', 
                'min_distance', 
                'nearest_idxs', 
                'location_on_contigs',
                'contig_segments',
            ]
            df0_subset = df0_subset[cols_to_store]
            df1_subset = df1_subset[cols_to_store]
            results0_by_gene[gene_idx].append(df0_subset)
            results1_by_gene[gene_idx].append(df1_subset)

    results0_by_gene = {idx: pd.concat(r) for idx, r in results0_by_gene.items()}
    results1_by_gene = {idx: pd.concat(r) for idx, r in results1_by_gene.items()}

    for df in results0_by_gene.values():
        df['nearest_idxs'] = df['nearest_idxs'].map(eval)
        df['location_on_contigs'] = df['location_on_contigs'].map(eval)
        df['contig_segments'] = df['contig_segments'].map(eval)

    for df in results1_by_gene.values():
        df['nearest_idxs'] = df['nearest_idxs'].map(eval)
        df['location_on_contigs'] = df['location_on_contigs'].map(eval)
        df['contig_segments'] = df['contig_segments'].map(eval)

    # if SAVE_DFS:
    #     for gidx, df in results0_by_gene.items():
    #         df.to_csv(f"{OUTDIR}/dataframes/df_results0_gene{gidx}.csv")
    #     for gidx, df in results1_by_gene.items():
    #         df.to_csv(f"{OUTDIR}/dataframes/df_results1_gene{gidx}.csv")

In [ ]:
results0_by_gene[0]

In [ ]:
results1_by_gene[0]

In [ ]:
fig, ax = plt.subplots(1, 1)

genome_lengths = [np.sum(x) for x in contig_lengths]
ax.hist(np.log10(genome_lengths), bins=50, log=True)

ax.set_title("Genome sizes")
ax.set_xlabel("$\\log_{10}$ genome length");
ax.set_ylabel("Count");

plt.savefig(f"{IMGDIR}/genome_sizes.pdf")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=None)

ax.loglog(
    genome_lengths, 
    [np.median(x) for x in contig_lengths], 
    '.',
    alpha=0.5
)

ax.set_title("Genome contig structure")
ax.set_xlabel("# contigs in genome")
ax.set_ylabel("median contig length");
plt.savefig(f"{IMGDIR}/genome_contig_structure.pdf")

In [ ]:
pd.concat([results0_by_gene[1].min_distance.value_counts(), 
           results1_by_gene[1].min_distance.value_counts()], axis=1)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=None)

width=0.4

genes2plot = np.array([0, 1, 2, 3, 4, 5], dtype=int)

vcs0 = [results0_by_gene[g].min_distance.value_counts() for g in genes2plot]
vcs1 = [results1_by_gene[g].min_distance.value_counts() for g in genes2plot]
bardata0 = np.array([
    [vcs0[g].get(i, 0) for i in range(nmut_thresh + 1)] for g in genes2plot
]).T
bardata1 = np.array([
    [vcs1[g].get(i, 0) for i in range(nmut_thresh + 1)] for g in genes2plot
]).T

# Each column of bardata{_} is a gene to plot as a single bar
xvals = np.arange(len(genes2plot))
bottoms0 = np.zeros(bardata0.shape[1])
bottoms1 = np.zeros(bardata1.shape[1])
for i in range(bardata0.shape[0]):
    b0 = ax.bar(
        xvals - width/2, bardata0[i], width*0.95, 
        bottom=bottoms0, 
    )
    b1 = ax.bar(
        xvals + width/2, bardata1[i], width*0.95, 
        bottom=bottoms1, 
        label=str(i),
        color=b0[0].get_facecolor()
    )
    bottoms0 += bardata0[i]
    bottoms1 += bardata1[i]
    
ax.set_xticks(range(len(genes2plot)), labels=[gene_map[i] for i in range(len(genes2plot))])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="dist from ref")

ax.set_title("Freq. of distances between query matches and reference")
ax.set_xlabel("Gene")
ax.set_ylabel("Count")

plt.savefig(f"{IMGDIR}/distfreq_barplot.pdf", bbox_inches="tight")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

fig, ax = plt.subplots(1, 1, figsize=(8,3))

width=0.4

genes2plot = np.arange(0,20)

vcs0 = [results0_by_gene[g].min_distance.value_counts() for g in genes2plot]
vcs1 = [results1_by_gene[g].min_distance.value_counts() for g in genes2plot]
bardata0 = np.array([
    [vcs0[g].get(i, 0) for i in range(nmut_thresh + 1)] for g in genes2plot
]).T
bardata1 = np.array([
    [vcs1[g].get(i, 0) for i in range(nmut_thresh + 1)] for g in genes2plot
]).T

combined_data = np.zeros([bardata0.shape[0], 3*bardata0.shape[1]], dtype=int)
combined_data[:,3*np.arange(bardata0.shape[1])] = bardata0
combined_data[:,3*np.arange(bardata0.shape[1]) + 1] = bardata1

combined_data = np.log10(combined_data)
# Each column of bardata{_} is a gene to plot as a single bar


combined_data.shape

# print(combined_data)
# colors = ['red', 'white', 'blue']
# cmap = LinearSegmentedColormap.from_list('red_white_blue', colors)
# norm = TwoSlopeNorm(
#     vmin=-combined_data.max(), vcenter=1, vmax=combined_data.max()
# )
sc = ax.pcolor(
    combined_data, 
    # cmap=cmap, 
    # norm=norm,
)

ax.set_xticks(
    3 * np.arange(len(genes2plot)) + 1,
    labels=[gene_map[i] for i in genes2plot], rotation=45
)
ax.set_yticks(
    0.5 + np.arange(nmut_thresh + 1), 
    labels=[str(i) for i in range(nmut_thresh + 1)]
)

ax.axhline(nmut_thresh + 1, color='r')

ax.set_xlabel("Gene")
ax.set_ylabel("num mutations")



cbar = fig.colorbar(sc, label="$\\log_{10}$ count")
# cbar.ax.set_title(cbar_title, size=cbar_titlefontsize)
# cbar.ax.tick_params(labelsize=cbar_ticklabelsize)

    
# ax.set_xticks(range(len(genes2plot)), labels=[gene_map[i] for i in range(len(genes2plot))])
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_title("Freq. of distances between query matches and reference")

plt.savefig(f"{IMGDIR}/distfreq_gridplot.pdf", bbox_inches="tight")

## Additional processing

Now subset the loaded data to only include matches less than or equal to the threshold.

In [ ]:
print(results0_by_gene[0]['min_distance'].max())
print(results1_by_gene[0]['min_distance'].max())

for i, df in results0_by_gene.items():
    results0_by_gene[i] = df[df['min_distance'] <= nmut_thresh]
for i, df in results1_by_gene.items():
    results1_by_gene[i] = df[df['min_distance'] <= nmut_thresh]

print(results0_by_gene[0]['min_distance'].max())
print(results1_by_gene[0]['min_distance'].max())

In [ ]:
# For each genome row, remove contig sequences containing underscores

# [[s for s in gene_idx_to_multimatches_rows0[0]['contig_segments'].values[i] if "_" not in s] for i in range(len(gene_idx_to_multimatches_rows0[0]))]

# def filter_underscores(contig_list):
#     return [s for s in contig_list if "_" not in s]

# for gene_idx, df in results0_by_gene.items():
#     wtseq = wtsequences[gene_idx]
#     nmatches = df['contig_segments'].map(len)
#     multimatched_screen = nmatches > 1
#     multimatched_rows = df[multimatched_screen]
#     df.loc[multimatched_screen, 'contig_segments'] = df[multimatched_screen]['contig_segments'].apply(
#         filter_underscores
#     )
#     results0_by_gene[gene_idx] = multimatched_rows

# for gene_idx, df in results1_by_gene.items():
#     wtseq = wtsequences[gene_idx]
#     nmatches = df['contig_segments'].map(len)
#     multimatched_screen = nmatches > 1
#     multimatched_rows = df[multimatched_screen]
#     df.loc[multimatched_screen, 'contig_segments'] = df[multimatched_screen]['contig_segments'].apply(
#         filter_underscores
#     )
#     results1_by_gene[gene_idx] = multimatched_rows

# # gene_idx_to_singlematch_rows1 = {}
# # gene_idx_to_multimatches_rows1 = {}
# # for gene_idx, df in results1_by_gene.items():
# #     wtseq = wtsequences[gene_idx]
# #     nmatches = df['contig_segments'].map(len)
# #     multimatched_screen = nmatches > 1
# #     multimatched_rows = df[multimatched_screen]
# #     gene_idx_to_multimatches_rows1[gene_idx] = multimatched_rows
# #     gene_idx_to_singlematch_rows1[gene_idx] = df[~multimatched_screen]


In [ ]:
# Now for each gene, scan through the contig segments for multiple matches

gene_idx_to_singlematch_rows0 = {}
gene_idx_to_multimatches_rows0 = {}
for gene_idx, df in results0_by_gene.items():
    wtseq = wtsequences[gene_idx]
    nmatches = df['contig_segments'].map(len)
    multimatched_screen = nmatches > 1
    multimatched_rows = df[multimatched_screen]
    gene_idx_to_multimatches_rows0[gene_idx] = multimatched_rows
    gene_idx_to_singlematch_rows0[gene_idx] = df[~multimatched_screen]

gene_idx_to_singlematch_rows1 = {}
gene_idx_to_multimatches_rows1 = {}
for gene_idx, df in results1_by_gene.items():
    wtseq = wtsequences[gene_idx]
    nmatches = df['contig_segments'].map(len)
    multimatched_screen = nmatches > 1
    multimatched_rows = df[multimatched_screen]
    gene_idx_to_multimatches_rows1[gene_idx] = multimatched_rows
    gene_idx_to_singlematch_rows1[gene_idx] = df[~multimatched_screen]


In [ ]:
gene_idx_to_singlematch_rows0[0]

In [ ]:
gene_idx_to_single_match_intersection_df = {}
for gidx in gene_idx_to_singlematch_rows0:
    df0 = gene_idx_to_singlematch_rows0[gidx]
    df1 = gene_idx_to_singlematch_rows1[gidx]
    intersection = pd.merge(df0, df1, on='genome_fpath')
    intersection['nearest_idxs_x'] = intersection['nearest_idxs_x'].map(lambda x: x[0])
    intersection['nearest_idxs_y'] = intersection['nearest_idxs_y'].map(lambda x: x[0])
    intersection['location_on_contigs_x'] = intersection['location_on_contigs_x'].map(lambda x: x[0])
    intersection['location_on_contigs_y'] = intersection['location_on_contigs_y'].map(lambda x: x[0])
    intersection['contig_segments_x'] = intersection['contig_segments_x'].map(lambda x: x[0])
    intersection['contig_segments_y'] = intersection['contig_segments_y'].map(lambda x: x[0])
    # intersection = intersection[(intersection['nearest_idxs_x'].map(len) < 2) & (intersection['nearest_idxs_y'].map(len) < 2)]
    # intersection = intersection[(intersection['contig_segments_x'].map(lambda x: "_" not in x)) & (intersection['contig_segments_y'].map(lambda x: "_" not in x))]
    print(gidx, gene_map[gidx], len(intersection))
    gene_idx_to_single_match_intersection_df[gidx] = intersection


In [ ]:
gene_idx_to_single_match_intersection_df[0]

In [ ]:
# Get sequences arrays 

gene_idx_to_seq_arrays_x = {}
gene_idx_to_seq_arrays_y = {}
for gidx, df in gene_idx_to_single_match_intersection_df.items():
    contigs_x = df['contig_segments_x'].values
    contigs_y = df['contig_segments_y'].values
    contigs_arr_x = np.array([gene_seq_to_array(s.upper()) for s in contigs_x])
    contigs_arr_y = np.array([gene_seq_to_array(s.upper()) for s in contigs_y])
    gene_idx_to_seq_arrays_x[gidx] = contigs_arr_x
    gene_idx_to_seq_arrays_y[gidx] = contigs_arr_y


In [ ]:
gene_idx_to_seq_arrays_x[0]


In [ ]:
gene_idx_to_seq_arrays_y[0]

## Save results

In [ ]:
# Save dataframes without sequences

if SAVE_DFS:
    for gidx, df in results0_by_gene.items():
        df.to_csv(f"{OUT_DF_DIR}/df_results0_gene{gidx}.csv")
    for gidx, df in results1_by_gene.items():
        df.to_csv(f"{OUT_DF_DIR}/df_results1_gene{gidx}.csv")

In [ ]:
# Save sequences in array format

if SAVE_DFS:
    for gidx, seqs0 in gene_idx_to_seq_arrays_x.items():
        np.save(f"{OUT_SEQ_DIR}/query_matches0_gene{gidx}.npy", seqs0)
    for gidx, seqs1 in gene_idx_to_seq_arrays_y.items():
        np.save(f"{OUT_SEQ_DIR}/query_matches1_gene{gidx}.npy", seqs1)


In [ ]:
# from natvar.helpers import NT_MAP

# num_unspecified_nts = np.sum(gene_idx_to_seq_arrays0[0] == NT_MAP['N'], axis=0)
# num_empty_nts = np.sum(gene_idx_to_seq_arrays0[0] == NT_MAP['_'], axis=0)
# print(num_unspecified_nts)
# print(num_empty_nts)

# def compute_entropy_by_position(seqarr):
#     num_unspecified_nts = np.sum(seqarr == NT_MAP['N'], axis=0)
#     num_empty_nts = np.sum(seqarr == NT_MAP['_'], axis=0)
#     num_exclude = num_empty_nts + num_unspecified_nts
#     ent = np.zeros(seqarr.shape[1])
#     for i in range(4):
#         p = np.sum(seqarr == i, axis=0) / (seqarr.shape[0] - num_exclude)
#         ent -= p * np.where(p == 0, 0, np.log2(p))
#     return ent


# idx = 23

# fig, ax = plt.subplots(1, 1)

# ax.plot(
#     compute_entropy_by_position(gene_idx_to_seq_arrays0[idx]), '.',
#     label="front query",  
# )
# ax.plot(
#     compute_entropy_by_position(gene_idx_to_seq_arrays1[idx]), '.',
#     label="back query",
# )

# ax.legend()
# ax.set_title(f"Gene {gene_map[idx]}")
# ax.set_xlabel("Position")
# ax.set_ylabel("Entropy");

# def identify_gene_variants(seqs):
#     return